### Create SMET Forcing Files from HRRR-AK Data: Mount Roberts Tram, WY2026 — Lapse-rate Corrected

Copy of the HRRR-only WY2026 notebook, using the temperature correction from
`2_create_snowpack_smet_forcing_hrrrak_lapserate_tram_WY2020-WY2025.ipynb`.
Uses the icefield-domain annual UTM dataset and the same nearest grid cell as
the existing TRAM WY2020–WY2025 extraction. Station elevation remains 529 m;
**6°C/km** temperature lapse rate is applied using the historical grid elevation
637.39 m. The correction is `-0.006 × (529 - 637.39) = +0.65034 K`.
Only air temperature changes; humidity, precipitation, wind, radiation, pressure,
and timestamps retain the HRRR-only values. No precipitation scaling or station
observations are used.

created by Cassie Lumbrazo\
updated: November 2026\
run location: UAS linux\
python environment: **rasterio**

Output: `/home/cassie/python/models/run_snowpack/sites/tram/input/hrrrak_lapserate_WY2026/hrrrak_lapserate_tram_WY2026.smet`

**Available coverage:** October 1, 2025 05:00 through September 29, 2026
04:00 UTC (8,712 hourly records). September 29–30 initialization folders are
missing, leaving 48 hours unavailable under the existing initialization-year /
forecast-valid-time convention. No terminal hours are invented or filled.
This notebook creates forcing only; it does not run SNOWPACK.

A matching empty `.sno` initial profile is also created beside the `.smet` file.


In [ ]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme()
plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy

import rioxarray
import rasterio 
import cfgrib
import os

import geopandas as gpd

from zoneinfo import ZoneInfo
from pathlib import Path

In [ ]:
Path.cwd()


So actually this python library exists: PySMET: https://snowpat.slf.ch/indexsmet/

but I am going to create my own script for now... 

## Load the HRRR Dataset, 

The WY2026 source is a full spatial grid rather than an existing site file.
Project the TRAM station coordinates to EPSG:32608 and select the nearest cell,
following `icefield_domain/2_merge_netcdfs.ipynb`. Load only that cell into memory.


In [ ]:
from pyproj import Transformer

input_file = Path("/hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2026_utm.nc")
lat_tram, lon_tram, elevation_tram = 58.2971, -134.386, 529
station_x, station_y = Transformer.from_crs(
    "EPSG:4326", "EPSG:32608", always_xy=True
).transform(lon_tram, lat_tram)

with xr.open_dataset(input_file) as source_grid:
    ds = source_grid.sel(x=station_x, y=station_y, method="nearest").load()
ds.rio.write_crs("EPSG:32608", inplace=True)

# Retain the annual files' initialization-year / forecast-valid-time convention.
expected_times = pd.date_range("2025-10-01 05:00", "2026-10-01 04:00", freq="1h")
times = pd.DatetimeIndex(ds.time.values)
if not times.is_unique or not times.is_monotonic_increasing:
    raise ValueError("Source timestamps must be unique and increasing")
if len(times.difference(expected_times)):
    raise ValueError("Source timestamps fall outside WY2026 forecast-valid coverage")
missing_times = expected_times.difference(times)
print(f"Selected HRRR cell: x={float(ds.x):.3f}, y={float(ds.y):.3f}")
print(f"Available: {len(times)} hours, {times[0]} through {times[-1]} UTC")
print(f"Unavailable hours: {len(missing_times)} (no terminal gap filling)")
ds


In [ ]:
# print the first few timestamps to see the time resolution
print(ds.time.values[:10])

In [ ]:
# check the crs 
print(ds.rio.crs)

# The Workflow Below, 

HRRR (NetCDF / xarray)\
   → your Python script\
   → SMET (ASCII)\
   → SNOWPACK


In [ ]:
# ds['temp'].plot() # 2-m surface temperature in K, need in K
# ds['rh'].plot() # 2-m relative humidity in %, need in fraction (0-1)
# ds['10u'].plot() # 10-m u wind component in m/s, need in m/s
# ds['10v'].plot() # 10-m v wind component in m/s, need in m/s
# ds['swrad'].plot() # surface downward shortwave radiation in W/m2, need in W/m2
# ds['lwrad'].plot() # surface downward longwave radiation in W/m2, need in W/m2
ds['precip_rate'].plot() # precipitation rate in kg/m2s, need in mm per timestep
# ds['pressure'].plot() # surface pressure in Pa, need in Pa

### Precipitation Variables from HRRR-AK

`ds.precip_rate` is the precipiation rate\
units: kg/m²/s\
range: 0–0.0035

In [ ]:
ds.precip_rate.attrs

#### MeteoIO Variable Names 
| MeteoIO name | Meaning                      | Required?            | Expected units              |
| ------------ | ---------------------------- | -------------------- | --------------------------- |
| `TA`         | Air temperature              | yes                  | °K                          |
| `RH`         | Relative humidity            | yes                  | fraction (0-1)              |
| `VW`         | Wind speed                   | yes                  | m s⁻¹                       |
| `DW`         | Wind direction               | optional             | degrees (0–360, from North) |
| `ISWR`       | Incoming shortwave radiation | strongly recommended | W m⁻²                       |
| `ILWR`       | Incoming longwave radiation  | strongly recommended | W m⁻²                       |
| `PSUM`       | Precipitation (increment)    | yes                  | mm (per timestep)           |
| `P`          | Air pressure                 | optional             | Pa                          |


HRRR to MeteIO Variable Mapping 

In [ ]:
# --- Unit expectations (SMET / MeteoIO) ---
# TA    : Kelvin
# RH    : fraction (0–1)
# VW    : m s-1
# DW    : degrees (meteorological)
# ISWR : W m-2
# ILWR : W m-2
# PSUM : mm per timestep
# P     : Pa

# Create SMET File

In [ ]:
from pathlib import Path
from pyproj import Transformer

This version dealing with the timestepping, and precipiation units over that timestep

In [ ]:
lat_tram, lon_tram, elevation_tram = 58.2971, -134.386, 529


In [ ]:
# =========================
# USER SETTINGS
# =========================
OUTFILE = Path("/home/cassie/python/models/run_snowpack/sites/tram/input/hrrrak_lapserate_WY2026/hrrrak_lapserate_tram_WY2026.smet")

STATION_ID = "hrrrak_lapserate_tram_WY2026"
STATION_NAME = "HRRR-AK TRAM Point Forcing - Lapse-rate Corrected Temperature"
LAT = lat_tram
LON = lon_tram
ELEV = elevation_tram

# Match the established TRAM lapse-rate notebook (6 degrees C per km).
HRRR_ELEV = 637.39
LAPSE_RATE = 0.006  # degrees C per metre; temperature increments equal kelvin
if not np.allclose(ds.orog.values, HRRR_ELEV, rtol=0, atol=0.01):
    raise ValueError("Source grid elevation differs from the historical TRAM cell")
elevation_difference = ELEV - HRRR_ELEV
temperature_correction = -LAPSE_RATE * elevation_difference
print(f"Temperature correction: {temperature_correction:+.5f} K")
EPSG = 32608
TZ = 0
NODATA = -999

# -------------------------
# SIMULATION TIMESTEP
# -------------------------
# Default: 1 hour
TIMESTEP = "1h"      # pandas-style string: "1h", "30min", "15min" H is deprecated, use lowercase
TIMESTEP_SECONDS = pd.to_timedelta(TIMESTEP).total_seconds()

# =========================
# COORDINATE TRANSFORM
# =========================
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32608",
    always_xy=True
)

EASTING, NORTHING = transformer.transform(LON, LAT)

# =========================
# BUILD DATAFRAME
# =========================
time = pd.to_datetime(ds.time.values)
df = pd.DataFrame(index=time)

# Base variables
df["TA"] = ds.temp.values + temperature_correction
df["RH"] = (ds.rh.values / 100.0)
df["ISWR"] = ds.swrad.values
df["ILWR"] = ds.lwrad.values
df["PS"] = ds.pressure.values

# ---- Wind speed & direction ----
u = ds['10u'].values
v = ds['10v'].values

df["VW"] = np.sqrt(u**2 + v**2)
df["DW"] = (270.0 - np.degrees(np.arctan2(v, u))) % 360.0

# ---- Precipitation conversion ----
# precip_rate: kg/m²/s → mm per timestep
df["PSUM"] = ds.precip_rate.values * TIMESTEP_SECONDS

# =========================
# RESAMPLE TO MODEL TIMESTEP
# =========================
# For energy variables → mean
mean_vars = ["TA", "RH", "VW", "DW", "ISWR", "ILWR", "PS"]

# For precipitation → sum
sum_vars = ["PSUM"]

df_mean = df[mean_vars].resample(TIMESTEP).mean()
df_sum = df[sum_vars].resample(TIMESTEP).sum(min_count=1)

df = pd.concat([df_mean, df_sum], axis=1)

# Keep missing precipitation as NODATA rather than interpreting it as zero.
if not np.isfinite(df.to_numpy()).all():
    raise ValueError("Missing or nonfinite forcing values; inspect before writing SMET")
if not df["RH"].between(0, 1).all():
    raise ValueError("Relative humidity outside 0–1")
if not df["DW"].between(0, 360, inclusive="left").all():
    raise ValueError("Wind direction outside 0–360 degrees")
if (df[["VW", "ISWR", "ILWR", "PSUM"]] < 0).any().any():
    raise ValueError("Negative wind, radiation, or precipitation forcing")
if not df.index.equals(times):
    raise ValueError("Resampling altered the available hourly timestamps")
df = df.fillna(NODATA)

# =========================
# WRITE SMET
# =========================
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUTFILE, "w") as f:
    f.write("SMET 1.1 ASCII\n")
    f.write("[HEADER]\n")

    f.write(f"station_id       = {STATION_ID}\n")
    f.write(f"station_name     = {STATION_NAME}\n")
    f.write(f"latitude         = {LAT:.6f}\n")
    f.write(f"longitude        = {LON:.6f}\n")
    f.write(f"easting          = {EASTING:.3f}\n")
    f.write(f"northing         = {NORTHING:.3f}\n")
    f.write(f"epsg             = {EPSG}\n")
    f.write(f"altitude         = {ELEV}\n")
    f.write(f"nodata           = {NODATA}\n")
    f.write(f"tz               = {TZ}\n")

    f.write("source           = HRRR-AK point extraction; temperature lapse-rate corrected to site elevation\n")
    f.write(f"lapse_rate       = {LAPSE_RATE:.4f} degC m-1\n")
    f.write(f"hrrr_elevation   = {HRRR_ELEV:.2f} m\n")
    f.write(f"site_elevation   = {ELEV:.2f} m\n")
    f.write(f"ta_correction    = {temperature_correction:.5f} K\n")
    f.write("fields           = timestamp TA RH VW DW ISWR ILWR PSUM PS\n")
    f.write("[DATA]\n")

    for t, row in df.iterrows():
        f.write(
            f"{t.strftime('%Y-%m-%dT%H:%M')} "
            f"{row.TA:8.2f} {row.RH:6.3f} "
            f"{row.VW:6.2f} {row.DW:6.1f} "
            f"{row.ISWR:8.1f} {row.ILWR:8.1f} "
            f"{row.PSUM:7.3f} {row.PS:8.1f}\n"
        )
print(f"Wrote {len(df)} hourly lapse-rate corrected HRRR records to {OUTFILE}")
print(f"Total available-period precipitation: {df.PSUM.sum():.3f} mm")


In [ ]:
# Read back and verify the forcing file, including the serialized precision.
with OUTFILE.open() as f:
    lines = f.readlines()
data_start = next(i for i, line in enumerate(lines) if line.strip() == "[DATA]") + 1
columns = ["timestamp", "TA", "RH", "VW", "DW", "ISWR", "ILWR", "PSUM", "PS"]
written = pd.read_csv(OUTFILE, skiprows=data_start, sep=r"\s+", names=columns,
                      parse_dates=["timestamp"]).set_index("timestamp")
assert written.index.equals(df.index)
assert len(written) == len(ds.time)
assert not written.isna().any().any()
for field, decimals in {"TA": 2, "RH": 3, "VW": 2, "DW": 1,
                        "ISWR": 1, "ILWR": 1, "PSUM": 3, "PS": 1}.items():
    np.testing.assert_allclose(written[field], df[field], rtol=0,
                               atol=0.5 * 10 ** (-decimals) + 1e-10)
print(f"Verified {len(written)} SMET records, timestamps, fields, and unit conversions.")
written.head()

# Independent comparison with the uncorrected forcing created previously.
raw_path = OUTFILE.parent.parent / "hrrrak_WY2026/hrrrak_tram_WY2026.smet"
with raw_path.open() as f:
    raw_lines = f.readlines()
raw_start = next(i for i, line in enumerate(raw_lines) if line.strip() == "[DATA]") + 1
raw_written = pd.read_csv(raw_path, skiprows=raw_start, sep=r"\s+", names=columns,
                          parse_dates=["timestamp"]).set_index("timestamp")
assert written.index.equals(raw_written.index)
pd.testing.assert_frame_equal(written.drop(columns="TA"), raw_written.drop(columns="TA"))
np.testing.assert_allclose(written.TA - raw_written.TA, temperature_correction,
                           rtol=0, atol=0.010000001)
print("Verified: only TA differs from the raw SMET; all other fields/timestamps match.")


## Create the empty SNOWPACK initial profile (.sno)

Use the TRAM terrain, soil, and canopy parameters from the existing forcing
notebooks: 38° slope, 0° aspect, no initial snow or soil layers. Match the station
metadata and filename to the forcing file. Following the existing lapse-rate
notebook, `ProfileDate` is the first forcing timestamp plus one model timestep
(October 1, 2025 at 06:00 UTC for these inputs).


In [ ]:
# ============================================================
# CREATE EMPTY .SNO FILE
# ============================================================

from datetime import timedelta

# ============================================================
# OUTPUT FILE
# ============================================================

OUTFILE_SNO = OUTFILE.with_suffix(".sno")

# ============================================================
# PROFILE DATE
# ============================================================

# Use the first timestamp in the forcing dataframe
# after resampling.
start_time = df.index.min()

# SNOWPACK commonly initializes one timestep after start
profile_date = pd.to_datetime(start_time) + pd.to_timedelta(TIMESTEP)

profile_date_str = profile_date.strftime("%Y-%m-%dT%H:%M")

# ============================================================
# OPTIONAL SITE PARAMETERS
# ============================================================

SLOPE_ANGLE = 38.0
SLOPE_AZI   = 0.0

SOIL_ALBEDO = 0.09
BARE_SOIL_Z0 = 0.200

CANOPY_HEIGHT = 0.00
CANOPY_LAI = 0.00
CANOPY_DIRECT_THROUGHFALL = 1.00

WIND_SCALING_FACTOR = 1.00

# ============================================================
# WRITE SNO FILE
# ============================================================

with open(OUTFILE_SNO, "w") as f:

    f.write("SMET 1.1 ASCII\n")
    f.write("[HEADER]\n")

    # --------------------------------------------------------
    # SITE INFORMATION
    # --------------------------------------------------------

    f.write(f"station_id       = {STATION_ID}\n")
    f.write(f"station_name     = {STATION_NAME}\n")

    f.write(f"latitude         = {LAT:.6f}\n")
    f.write(f"longitude        = {LON:.6f}\n")

    f.write(f"easting          = {EASTING:.3f}\n")
    f.write(f"northing         = {NORTHING:.3f}\n")

    f.write(f"epsg             = {EPSG}\n")
    f.write(f"altitude         = {ELEV}\n")

    f.write(f"nodata           = {NODATA}\n")
    f.write(f"tz               = {TZ}\n")

    f.write("source           = HRRR-AK point extraction; temperature lapse-rate corrected to site elevation\n")
    f.write(f"lapse_rate       = {LAPSE_RATE:.4f} degC m-1\n")
    f.write(f"hrrr_elevation   = {HRRR_ELEV:.2f} m\n")
    f.write(f"site_elevation   = {ELEV:.2f} m\n")
    f.write(f"ta_correction    = {temperature_correction:.5f} K\n")

    # --------------------------------------------------------
    # SNOWPACK INITIALIZATION SETTINGS
    # --------------------------------------------------------

    f.write(f"ProfileDate      = {profile_date_str}\n")

    # Initial snow height
    f.write("HS_Last          = 0.0000\n")

    # Terrain
    f.write(f"SlopeAngle       = {SLOPE_ANGLE:.1f}\n")
    f.write(f"SlopeAzi         = {SLOPE_AZI:.1f}\n")

    # Empty snowpack / soil profile
    f.write("nSoilLayerData   = 0\n")
    f.write("nSnowLayerData   = 0\n")

    # Soil / canopy parameters
    f.write(f"SoilAlbedo       = {SOIL_ALBEDO:.2f}\n")
    f.write(f"BareSoil_z0      = {BARE_SOIL_Z0:.3f}\n")

    f.write(f"CanopyHeight     = {CANOPY_HEIGHT:.2f}\n")
    f.write(f"CanopyLeafAreaIndex = {CANOPY_LAI:.2f}\n")

    f.write(
        f"CanopyDirectThroughfall = "
        f"{CANOPY_DIRECT_THROUGHFALL:.2f}\n"
    )

    f.write(
        f"WindScalingFactor = "
        f"{WIND_SCALING_FACTOR:.2f}\n"
    )

    f.write("ErosionLevel     = 0\n")
    f.write("TimeCountDeltaHS = 0.000000\n")

    # --------------------------------------------------------
    # REQUIRED FIELD DEFINITIONS
    # --------------------------------------------------------

    f.write(
        "fields           = "
        "timestamp Layer_Thick T Vol_Frac_I "
        "Vol_Frac_W Vol_Frac_V Vol_Frac_S "
        "Rho_S Conduc_S HeatCapac_S "
        "rg rb dd sp mk mass_hoar ne CDot metamo\n"
    )

    # --------------------------------------------------------
    # EMPTY DATA SECTION
    # --------------------------------------------------------

    f.write("[DATA]\n")

print(f"\nSNO file written to:\n{OUTFILE_SNO}")

In [ ]:
# Check station metadata, initialization date, and the empty profile.
sno_lines = OUTFILE_SNO.read_text().splitlines()
sno_header = dict(line.split("=", 1) for line in sno_lines if "=" in line)
sno_header = {key.strip(): value.strip() for key, value in sno_header.items()}
for key, expected in {"station_id": STATION_ID, "station_name": STATION_NAME,
                      "epsg": str(EPSG), "tz": str(TZ)}.items():
    assert sno_header[key] == expected
assert float(sno_header["altitude"]) == ELEV
assert pd.Timestamp(sno_header["ProfileDate"]) == df.index.min() + pd.to_timedelta(TIMESTEP)
assert float(sno_header["HS_Last"]) == 0
assert int(sno_header["nSoilLayerData"]) == 0
assert int(sno_header["nSnowLayerData"]) == 0
assert len(sno_header["fields"].split()) == 19
assert sno_lines[-1] == "[DATA]"
print(f"Verified empty SNO profile: {STATION_ID}, {sno_header['ProfileDate']} UTC")
